# Практика 8. Групування і зведені таблиці

**Студент:** Yurii_huz  
**Група:** IT-41  
**Варіант:** 1, Київ  
**Середньорічна температура:** 9.5°C  
**Сезонна амплітуда:** 14°C

Мета: застосувати `groupby`, `.agg()`, `pivot_table()` і `crosstab()` до довгого кліматичного набору даних.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
base_temp = 9.5
amplitude = 14
city = "Київ"
rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        rows.append({"місто": city, "рік": year, "місяць": month, "температура": round(base_temp + seasonal + noise, 1)})
climate = pd.DataFrame(rows)
print(climate.head())
print("Форма набору:", climate.shape)

## Завдання 1. `groupby` і `.agg()` за роками

Групування виконує split-apply-combine: спочатку розділяє рядки за роками, потім застосовує статистики до кожної групи, а наприкінці об'єднує результати в таблицю.

In [ ]:
year_stats = climate.groupby("рік")["температура"].agg(["mean", "min", "max"])
print(year_stats)

Середні значення температури за роками можуть трохи відрізнятися через випадковий шум. Якщо вони не зростають послідовно від 2021 до 2024 року, помітного тренду потепління в цьому синтетичному наборі немає, а коливання можна вважати випадковими.

## Завдання 2. `groupby` і `.agg()` за місяцями

In [ ]:
month_stats = climate.groupby("місяць")["температура"].agg(["mean", "std"])
print(month_stats)
most_unstable_month = month_stats["std"].idxmax()
print("Найбільший розкид між роками має місяць:", most_unstable_month)
print("Його std:", month_stats.loc[most_unstable_month, "std"])

Найнестабільнішим може виявитися перехідний місяць, наприклад квітень або жовтень: у такі періоди погода швидше змінюється, тому випадкове відхилення сильніше впливає на результат. Точний місяць визначається обчисленням `idxmax()` для отриманої таблиці.

## Завдання 3. `pivot_table()`

Зведена форма зручніша для візуального порівняння місяців між роками. Довга tidy-форма зручніша для `groupby`, фільтрації та побудови графіків, бо рік і місяць залишаються значеннями окремих змінних.

In [ ]:
temperature_pivot = climate.pivot_table(index="місяць", columns="рік", values="температура", aggfunc="mean")
print(temperature_pivot)

## Завдання 4. Похідні категорії і `crosstab()`

In [ ]:
def season_by_month(month):
    if month in [12, 1, 2]:
        return "зима"
    if month in [3, 4, 5]:
        return "весна"
    if month in [6, 7, 8]:
        return "літо"
    return "осінь"

climate["сезон"] = climate["місяць"].apply(season_by_month)
climate["тепліше_за_середнє"] = climate["температура"] > base_temp
season_crosstab = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])
print(season_crosstab)

`crosstab()` показує кількість спостережень у кожній комбінації двох категорій. Літні місяці мають найвищу сезонну складову, тому очікується, що вони переважно або повністю потраплять у категорію `True`; випадковий шум може вплинути на прикордонні місяці. Для двох категоріальних змінних це компактна таблиця частот, подібна до результату `groupby(["сезон", "тепліше_за_середнє"]).size()`.

## Завдання 5. `pivot()` на власному наборі

In [ ]:
simple_pivot = climate.pivot(index="місяць", columns="рік", values="температура")
print(simple_pivot)
print("Кількість рядків на пару місяць/рік:", climate.groupby(["місяць", "рік"]).size().unique())

with_stations = pd.concat([climate.assign(станція="A"), climate.assign(станція="B")], ignore_index=True)
try:
    with_stations.pivot(index="місяць", columns="рік", values="температура")
except ValueError as error:
    print("Очікувана помилка після додавання двох станцій:", error)

`pivot()` працює на початковому наборі, бо кожній комбінації `місяць/рік` відповідає рівно один рядок. Після додавання двох станцій на кожну комбінацію припадають два значення, тому `pivot()` не знає, яке вибрати, і завершується помилкою про дублікати. У такій ситуації потрібно використовувати `pivot_table()` з агрегувальною функцією, наприклад `mean`.

## Письмові відповіді

1. `groupby()` реалізує split-apply-combine: розділяє дані на групи за значенням ключа, застосовує до кожної групи агрегат або іншу функцію, а потім об'єднує результати.

2. `pivot()` вимагає унікальної комбінації індексу і стовпця, тому не працює з дублікатами. `pivot_table()` додатково агрегує кілька значень через `aggfunc`, тому може працювати з повторними комбінаціями.

3. `crosstab()` рахує частоти комбінацій категоріальних змінних і повертає їх у табличній формі. `groupby(...).size()` повертає подібні частоти, але зазвичай у вигляді Series з багаторівневим індексом, яку за потреби треба розгорнути.

4. Tidy-формат зберігає кожну змінну в окремому стовпці, тому `groupby` і `pivot_table` можуть безпосередньо працювати з роком, місяцем і температурою. У wide-таблиці роки або місяці стали б частинами назв стовпців, що ускладнило б універсальне групування.

## Підсумок

Побудовано tidy-набір Києва за 2021–2024 роки, виконано групування за роками й місяцями, створено `pivot_table` і `crosstab`, а також показано різницю між успішним `pivot()` та помилкою на дубльованих комбінаціях.